# Project A: more synthetic corpora (M7)

The consolidation result (uniform replay beats the compaction gate, paired t(4) = 3.10) rests on
five corpora, data seeds 0 to 4. This notebook runs further corpora through the same three arms
that Table 4 reports: cascading, compaction-gated consolidation and uniform replay. Settings
match `a1_main.ipynb` with `DATASET = 'synthetic'`: 48 train and 48 eval trajectories of 120
turns, a Qwen2.5-1.5B compactor and a Qwen2.5-0.5B target.

Each seed writes its own run root, so seeds never share data, compaction events or adapters.
After each seed the three retention summaries are copied to `/kaggle/working/m7/seed_<n>/` and
zipped to `/kaggle/working/m7_seeds.zip`, so a session that times out keeps the finished seeds.

One corpus takes about 32 minutes on a T4 (seed 5: compaction 7, each sleep run 9, evaluation 8), so seeds 5 to 9 fit in one session. Set `DATA_SEEDS` to run a subset.


In [ ]:
import glob, os, subprocess, sys, zipfile

GIT_URL = 'https://github.com/rajul-kk/context-to-weights.git'
REPO = '/kaggle/working/myrios'
MARKER = 'baselines/cascading.py'

def looks_like_source(d):
    return os.path.exists(os.path.join(d, MARKER))

if not looks_like_source(REPO):
    src = None
    for d in sorted(glob.glob('/kaggle/input/*')):
        if looks_like_source(d):
            src = d
            break
        for z in sorted(glob.glob(os.path.join(d, '*.zip'))):
            os.makedirs(REPO, exist_ok=True)
            zipfile.ZipFile(z).extractall(REPO)
            if looks_like_source(REPO):
                src = REPO
                break
        if src:
            break
    if src and src != REPO:
        subprocess.run(['cp', '-r', src, REPO], check=True)
    if not looks_like_source(REPO) and GIT_URL:
        r = subprocess.run(['git', 'clone', GIT_URL, REPO], capture_output=True, text=True)
        print(r.stdout, r.stderr)
    assert looks_like_source(REPO), (
        'No source found. Either (a) run scripts/package_source.py locally, upload the zip '
        'as a Kaggle Dataset, and attach it via Add Input, or (b) set GIT_URL above. '
        f'Searched /kaggle/input/*, saw: {sorted(glob.glob("/kaggle/input/*"))}')

os.chdir(REPO)
sys.path.insert(0, REPO)
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'peft', 'accelerate', 'datasets'])

try:
    import importlib.metadata as _md
    _v = _md.version('torchao')
    if tuple(int(x) for x in _v.split('.')[:2]) < (0, 16):
        subprocess.run([sys.executable, '-m', 'pip', '-q', 'uninstall', '-y', 'torchao'])
        print(f'removed incompatible torchao {_v}')
except Exception:
    pass
exec(open('notebooks/_runner.py').read())
print('cwd', os.getcwd())
print('files', sorted(os.listdir('.'))[:10])
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
if not torch.cuda.is_available():
    print()
    print('=' * 68)
    print('NO GPU. Kaggle installed the CPU build of torch, so this session')
    print('has no accelerator attached. Everything below will be far too slow.')
    print()
    print('Fix: right panel -> Session options -> Accelerator -> GPU T4 x2,')
    print('then Run All again. The image swaps to a CUDA torch build on restart.')
    print('=' * 68)
else:
    print('gpu', torch.cuda.get_device_name(0),
          f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')



## Config


In [ ]:
import os, sys, time, json, shutil
REPO = '/kaggle/working/myrios'
os.chdir(REPO); sys.path.insert(0, REPO)
if 'run' not in dir():
    exec(open('notebooks/_runner.py').read())

DATA_SEEDS = [5, 6, 7, 8, 9]
CFG = 'configs/kaggle_synthetic.yaml'
COMPACTOR = 'Qwen/Qwen2.5-1.5B-Instruct'
TARGET = 'Qwen/Qwen2.5-0.5B-Instruct'
N_TRAIN, N_EVAL, N_TURNS = 48, 48, 120
OUT = '/kaggle/working/m7'
os.makedirs(OUT, exist_ok=True)


## Run


In [ ]:
def adapter_for(runs, tag):
    best = f'{runs}/sleep_{tag}/best/adapter'
    return best if os.path.exists(best) else f'{runs}/sleep_{tag}/latest/adapter'

def one_seed(seed):
    runs = f'/kaggle/working/artifacts/runs_synth_s{seed}'
    data = f'artifacts/data/synthetic_s{seed}'
    casc = f'{runs}/cascading'
    report = f'{runs}/report'
    over = f'data.dir={data} run_root={runs}'
    t0 = time.time()
    run(f'python data/generate_synthetic.py --n-train {N_TRAIN} --n-eval {N_EVAL} --n-turns {N_TURNS} --seed {seed} --out {data}')
    run(f'python baselines/cascading.py --config {CFG} --split both --out {casc} --set model.base={COMPACTOR} {over}')
    run(f'python eval/span_report.py --events {casc}/eval_events.jsonl --show 0 --out {casc}/eval_span_report.json')
    times = {'compaction': time.time() - t0}
    for method in ('compaction', 'uniform'):
        t = time.time()
        run(f'python sleep/loop.py --config {CFG} --method {method} --events {casc}/eval_events.jsonl '
            f'--val-events {casc}/train_events.jsonl --run-dir {runs}/sleep_{method} --resume --val-limit 64 '
            f'--set model.base={TARGET} {over}')
        times[f'sleep_{method}'] = time.time() - t
    t = time.time()
    for label, tag in (('cascading', None), ('ours', 'compaction'), ('uniform', 'uniform')):
        a = f'--adapter {adapter_for(runs, tag)}' if tag else ''
        run(f'python eval/retention.py --config {CFG} --contexts {casc}/eval_contexts.jsonl --label {label} '
            f'--out {report}/{label} {a} --set model.base={TARGET} {over}')
    times['eval'] = time.time() - t
    times['total'] = time.time() - t0
    dst = f'{OUT}/seed_{seed}'
    os.makedirs(dst, exist_ok=True)
    for label in ('cascading', 'ours', 'uniform'):
        shutil.copy(f'{report}/{label}.summary.json', dst)
    shutil.copy(f'{casc}/eval_span_report.json', dst)
    json.dump(times, open(f'{dst}/times.json', 'w'), indent=1)
    shutil.make_archive('/kaggle/working/m7_seeds', 'zip', OUT)
    rows = {l: json.load(open(f'{dst}/{l}.summary.json')) for l in ('cascading', 'ours', 'uniform')}
    print(f'seed {seed}: n_evicted {rows["uniform"]["n_evicted"]}  ' + '  '.join(
        f'{l} {r["evicted_accuracy"]:.3f}/{r["retention_accuracy"]:.3f}' for l, r in rows.items()))
    print(f'seed {seed}: minutes ' + json.dumps({k: round(v / 60, 1) for k, v in times.items()}), flush=True)

for s in DATA_SEEDS:
    one_seed(s)
